In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
CSV_PATH = "gen8ou-1825/gen8ou_1825_usage_timeseries.csv"

df = pd.read_csv(CSV_PATH)
df["usage_pct"] = df["usage"] * 100

df.head()

,month,pokemon,usage,raw_count,real_usage,total_battles,usage_pct
0,2021-01,Abomasnow,0.000186,8017.0,0.004444,1804169,0.01860
1,2021-03,Abomasnow,0.001671,6952.0,0.004270,1628081,0.16706
2,2021-04,Abomasnow,0.000376,4922.0,0.003520,1398223,0.03756
3,2021-05,Abomasnow,0.000371,4179.0,0.003105,1345731,0.03707
4,2021-06,Abomasnow,0.000116,5361.0,0.004011,1336739,0.01162


In [3]:
unique_pokemon = df["pokemon"].unique()
print(unique_pokemon)

['Abomasnow' 'Abra' 'Absol' 'Accelgor' 'Aegislash' 'Aerodactyl' 'Aggron'
 'Alakazam' 'Alcremie' 'Altaria' 'Amoonguss' 'Appletun' 'Araquanid'
 'Arcanine' 'Archeops' 'Arctovish' 'Arctozolt' 'Armaldo' 'Aron' 'Articuno'
 'Articuno-Galar' 'Audino' 'Aurorus' 'Avalugg' 'Azelf' 'Azumarill'
 'Barbaracle' 'Barraskewda' 'Basculin' 'Beartic' 'Beheeyem' 'Bellossom'
 'Bewear' 'Bisharp' 'Blacephalon' 'Blastoise' 'Blaziken' 'Blissey'
 'Boltund' 'Bouffalant' 'Braviary' 'Bronzong' 'Bulbasaur' 'Butterfree'
 'Buzzwole' 'Calyrex' 'Carbink' 'Carracosta' 'Celebi' 'Celesteela'
 'Centiskorch' 'Chandelure' 'Chansey' 'Charizard' 'Cherrim' 'Cinccino'
 'Cinderace' 'Clawitzer' 'Claydol' 'Clefable' 'Cloyster' 'Coalossal'
 'Cobalion' 'Cofagrigus' 'Combusken' 'Comfey' 'Conkeldurr' 'Copperajah'
 'Corsola' 'Corsola-Galar' 'Corviknight' 'Cosmoem' 'Cradily' 'Cramorant'
 'Crawdaunt' 'Cresselia' 'Crobat' 'Crustle' 'Cryogonal' 'Cursola'
 'Darmanitan' 'Decidueye' 'Dedenne' 'Delibird' 'Dhelmise' 'Diancie'
 'Diggersby' 'Diglett

In [ ]:
import requests
import time

# PokeAPI dex IDs for Sword/Shield
# 27 = galar, 28 = isle-of-armor, 29 = crown-tundra
SWSH_DEX_IDS = [27, 28, 29]

def get_dex_species(dex_id):
    url = f"https://pokeapi.co/api/v2/pokedex/{dex_id}/"
    r = requests.get(url)
    r.raise_for_status()
    return [e["pokemon_species"]["name"] for e in r.json()["pokemon_entries"]]

def is_fully_evolved(species_name):
    url = f"https://pokeapi.co/api/v2/pokemon-species/{species_name}/"
    r = requests.get(url)
    if r.status_code != 200:
        return False
    data = r.json()
    evo_chain_url = data["evolution_chain"]["url"]
    chain_r = requests.get(evo_chain_url)
    chain_r.raise_for_status()

    # Walk the chain and collect all terminal (fully evolved) species
    def get_terminals(chain):
        if not chain["evolves_to"]:
            return [chain["species"]["name"]]
        terminals = []
        for branch in chain["evolves_to"]:
            terminals.extend(get_terminals(branch))
        return terminals

    terminals = get_terminals(chain_r.json()["chain"])
    return species_name in terminals

# Collect all unique species across all three dexes
all_species = set()
for dex_id in SWSH_DEX_IDS:
    all_species.update(get_dex_species(dex_id))

print(f"Total unique species in Sword/Shield dexes: {len(all_species)}")

# Filter to fully evolved — this takes a while due to API calls
gen8_fully_evolved = []
for i, name in enumerate(sorted(all_species)):
    if is_fully_evolved(name):
        gen8_fully_evolved.append(name)
    time.sleep(0.3)  # be polite to the API
    if i % 50 == 0:
        print(f"  {i}/{len(all_species)} checked...")

print(f"\n{len(gen8_fully_evolved)} fully evolved Pokémon in Gen 8:")
print(gen8_fully_evolved)

Total unique species in Sword/Shield dexes: 584
  0/584 checked...
  50/584 checked...
  100/584 checked...
  150/584 checked...
  200/584 checked...
  250/584 checked...
  300/584 checked...
  350/584 checked...
  400/584 checked...
  450/584 checked...
  500/584 checked...
  550/584 checked...

300 fully evolved Pokémon in Gen 8:
['abomasnow', 'absol', 'accelgor', 'aegislash', 'aerodactyl', 'aggron', 'alakazam', 'alcremie', 'altaria', 'amoonguss', 'appletun', 'araquanid', 'arcanine', 'archeops', 'arctovish', 'arctozolt', 'armaldo', 'aromatisse', 'articuno', 'audino', 'aurorus', 'avalugg', 'azumarill', 'barbaracle', 'barraskewda', 'beartic', 'beheeyem', 'bellossom', 'bewear', 'blastoise', 'blissey', 'boltund', 'bouffalant', 'braviary', 'bronzong', 'butterfree', 'calyrex', 'carbink', 'carracosta', 'centiskorch', 'chandelure', 'charizard', 'cherrim', 'cinccino', 'cinderace', 'clawitzer', 'claydol', 'clefable', 'cloyster', 'coalossal', 'cobalion', 'cofagrigus', 'comfey', 'conkeldurr', 'c

In [ ]:

home_extras_fully_evolved = {
    "mewtwo", "mew", "celebi", "jirachi",
    "reshiram", "zekrom", "kyurem", "keldeo",
    "decidueye", "incineroar", "primarina",
    "solgaleo", "lunala", "necrozma",
    "marshadow", "zeraora", "melmetal", "magearna",
    "raikou", "entei", "suicune", "lugia", "ho-oh",
    "sceptile", "blaziken", "swampert",
    "latias", "latios", "kyogre", "groudon", "rayquaza",
    "uxie", "mesprit", "azelf",
    "dialga", "palkia", "heatran", "regigigas", "giratina", "cresselia",
    "victini", "tornadus", "thundurus", "landorus", "genesect",
    "xerneas", "yveltal", "zygarde", "diancie", "volcanion",
    "tapu-koko", "tapu-lele", "tapu-bulu", "tapu-fini",
    "nihilego", "buzzwole", "pheromosa", "xurkitree",
    "celesteela", "kartana", "guzzlord",
    "naganadel", "stakataka", "blacephalon",
}

gen8_fully_evolved = list(set(gen8_fully_evolved) | home_extras_fully_evolved)

print(f"Total fully evolved Gen 8 compatible: {len(gen8_fully_evolved)}")

Total fully evolved Gen 8 compatible: 364


In [6]:
csv_pokemon = set(df["pokemon"].str.lower().str.replace(" ", "-"))
gen8_set = set(gen8_fully_evolved)

not_in_csv = gen8_set - csv_pokemon

print(f"{len(not_in_csv)} Pokémon in Gen 8 list but not in CSV:\n")
for name in sorted(not_in_csv):
    print(name)

46 Pokémon in Gen 8 list but not in CSV:

aromatisse
dialga
dracovish
eternatus
garbodor
genesect
giratina
glalie
gourgeist
groudon
heatmor
ho-oh
kyogre
landorus
lugia
lunala
lunatone
marshadow
mewtwo
mr-rime
musharna
naganadel
oranguru
palkia
passimian
pheromosa
pinsir
raichu
rayquaza
reshiram
rotom
seaking
sirfetchd
skuntank
solgaleo
solrock
sudowoodo
throh
whiscash
wobbuffet
xerneas
yveltal
zacian
zamazenta
zekrom
zygarde


In [7]:
# All HOME-compatible extras (full lines, not just fully evolved)
home_extras_all = {
    "mewtwo", "mew", "celebi", "jirachi",
    "reshiram", "zekrom", "kyurem", "keldeo",
    "rowlet", "dartrix", "decidueye",
    "litten", "torracat", "incineroar",
    "popplio", "brionne", "primarina",
    "cosmog", "cosmoem", "solgaleo", "lunala", "necrozma",
    "marshadow", "zeraora", "meltan", "melmetal", "magearna",
    "raikou", "entei", "suicune", "lugia", "ho-oh",
    "treecko", "grovyle", "sceptile",
    "torchic", "combusken", "blaziken",
    "mudkip", "marshtomp", "swampert",
    "latias", "latios", "kyogre", "groudon", "rayquaza",
    "uxie", "mesprit", "azelf",
    "dialga", "palkia", "heatran", "regigigas", "giratina", "cresselia",
    "victini", "tornadus", "thundurus", "landorus", "genesect",
    "xerneas", "yveltal", "zygarde", "diancie", "volcanion",
    "tapu-koko", "tapu-lele", "tapu-bulu", "tapu-fini",
    "nihilego", "buzzwole", "pheromosa", "xurkitree",
    "celesteela", "kartana", "guzzlord",
    "poipole", "naganadel", "stakataka", "blacephalon",
}

# Fetch all species from the three in-game dexes
all_swsh_species = set()
for dex_id in SWSH_DEX_IDS:
    all_swsh_species.update(get_dex_species(dex_id))

# Combine
gen8_all_pokemon = all_swsh_species | home_extras_all

# Put into a sorted list (the "matrix")
gen8_matrix = sorted(gen8_all_pokemon)

print(f"Total Gen 8 compatible Pokémon: {len(gen8_matrix)}")
print(gen8_matrix)

Total Gen 8 compatible Pokémon: 664
['abomasnow', 'abra', 'absol', 'accelgor', 'aegislash', 'aerodactyl', 'aggron', 'alakazam', 'alcremie', 'altaria', 'amaura', 'amoonguss', 'anorith', 'appletun', 'applin', 'araquanid', 'arcanine', 'archen', 'archeops', 'arctovish', 'arctozolt', 'armaldo', 'aromatisse', 'aron', 'arrokuda', 'articuno', 'audino', 'aurorus', 'avalugg', 'axew', 'azelf', 'azumarill', 'azurill', 'bagon', 'baltoy', 'barbaracle', 'barboach', 'barraskewda', 'basculin', 'beartic', 'beheeyem', 'beldum', 'bellossom', 'bergmite', 'bewear', 'binacle', 'bisharp', 'blacephalon', 'blastoise', 'blaziken', 'blipbug', 'blissey', 'boldore', 'boltund', 'bonsly', 'bouffalant', 'bounsweet', 'braviary', 'brionne', 'bronzong', 'bronzor', 'budew', 'bulbasaur', 'buneary', 'bunnelby', 'butterfree', 'buzzwole', 'calyrex', 'carbink', 'carkol', 'carracosta', 'carvanha', 'caterpie', 'celebi', 'celesteela', 'centiskorch', 'chandelure', 'chansey', 'charizard', 'charjabug', 'charmander', 'charmeleon', 'c

In [8]:
csv_pokemon = set(df["pokemon"].str.lower().str.replace(" ", "-"))
gen8_matrix_set = set(gen8_matrix)

not_in_csv = gen8_matrix_set - csv_pokemon

print(f"{len(not_in_csv)} Pokémon in Gen 8 matrix but not in CSV:\n")
for name in sorted(not_in_csv):
    print(name)

308 Pokémon in Gen 8 matrix but not in CSV:

amaura
anorith
applin
archen
aromatisse
arrokuda
axew
azurill
bagon
baltoy
barboach
beldum
bergmite
binacle
blipbug
boldore
bonsly
bounsweet
brionne
bronzor
budew
buneary
bunnelby
carkol
carvanha
caterpie
charjabug
charmander
charmeleon
cherubi
chewtle
chinchou
clauncher
clefairy
cleffa
clobbopus
combee
corphish
corvisquire
cosmog
cottonee
croagunk
cubchoo
cubone
cufant
cutiefly
dartrix
darumaka
deino
dewpider
dialga
diglett
dottler
dracovish
dragonair
drakloak
dratini
dreepy
drifloon
drilbur
drizzile
duosion
duskull
dwebble
eevee
electabuzz
electrike
elekid
elgyem
espurr
eternatus
exeggcute
farfetchd
feebas
fletchinder
fletchling
fomantis
foongus
fraxure
frillish
gabite
garbodor
gastly
genesect
gible
giratina
glalie
gloom
golbat
goldeen
golett
goomy
gossifleur
gothita
gothorita
gourgeist
grookey
groudon
grovyle
growlithe
grubbin
gurdurr
hakamo-o
happiny
hatenna
hattrem
haunter
heatmor
helioptile
herdier
hippopotas
ho-oh
honedge
hoothoot
hor

In [9]:
# Fully evolved HOME extras only (same as before)
home_extras_fully_evolved = {
    "mewtwo", "mew", "celebi", "jirachi",
    "reshiram", "zekrom", "kyurem", "keldeo",
    "decidueye", "incineroar", "primarina",
    "solgaleo", "lunala", "necrozma",
    "marshadow", "zeraora", "melmetal", "magearna",
    "raikou", "entei", "suicune", "lugia", "ho-oh",
    "sceptile", "blaziken", "swampert",
    "latias", "latios", "kyogre", "groudon", "rayquaza",
    "uxie", "mesprit", "azelf",
    "dialga", "palkia", "heatran", "regigigas", "giratina", "cresselia",
    "victini", "tornadus", "thundurus", "landorus", "genesect",
    "xerneas", "yveltal", "zygarde", "diancie", "volcanion",
    "tapu-koko", "tapu-lele", "tapu-bulu", "tapu-fini",
    "nihilego", "buzzwole", "pheromosa", "xurkitree",
    "celesteela", "kartana", "guzzlord",
    "naganadel", "stakataka", "blacephalon",
}

# Combine PokeAPI-derived fully evolved with HOME extras
gen8_fully_evolved_matrix = sorted(set(gen8_fully_evolved) | home_extras_fully_evolved)

csv_pokemon = set(df["pokemon"].str.lower().str.replace(" ", "-"))

not_in_csv_final = set(gen8_fully_evolved_matrix) - csv_pokemon

print(f"Total fully evolved Gen 8 Pokémon: {len(gen8_fully_evolved_matrix)}")
print(f"\n{len(not_in_csv_final)} with zero usage not in CSV:\n")
for name in sorted(not_in_csv_final):
    print(name)

Total fully evolved Gen 8 Pokémon: 364

46 with zero usage not in CSV:

aromatisse
dialga
dracovish
eternatus
garbodor
genesect
giratina
glalie
gourgeist
groudon
heatmor
ho-oh
kyogre
landorus
lugia
lunala
lunatone
marshadow
mewtwo
mr-rime
musharna
naganadel
oranguru
palkia
passimian
pheromosa
pinsir
raichu
rayquaza
reshiram
rotom
seaking
sirfetchd
skuntank
solgaleo
solrock
sudowoodo
throh
whiscash
wobbuffet
xerneas
yveltal
zacian
zamazenta
zekrom
zygarde


In [10]:
import itertools

# Build rows for every missing pokemon across every month in the dataset
all_months = df["month"].unique()

zero_usage_rows = pd.DataFrame([
    {
        "month": month,
        "pokemon": name,
        "usage": 0.0,
        "raw_count": 0.0,
        "real_usage": 0.0,
        "total_battles": df[df["month"] == month]["total_battles"].iloc[0],
        "usage_pct": 0.0,
    }
    for name, month in itertools.product(not_in_csv_final, all_months)
])

df_extended = pd.concat([df, zero_usage_rows], ignore_index=True).sort_values(
    ["month", "pokemon"]
).reset_index(drop=True)

output_path = Path("gen8ou-1825/gen8ouExtras.csv")
df_extended.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(f"Original rows: {len(df)}")
print(f"Added rows:    {len(zero_usage_rows)}")
print(f"Total rows:    {len(df_extended)}")

Saved to gen8ou-1825\gen8ouExtras.csv
Original rows: 5489
Added rows:    1012
Total rows:    6501
